<!-- [TAG: lab-intro] -->
# Lab 8 — Comprehensive Algorithm Comparison
### MSDS 684 · Reinforcement Learning

Three families of reinforcement learning algorithms walk into LunarLander. One knows only tables. One ignores values entirely and bets on the policy. One builds a model of the world and hallucinates experience it never had.

This lab is about watching all three compete on the same problem, and understanding why the competition ends the way it does.

**The plan has four parts.**

The **first part** is setup — imports, environment description, and a sanity check before anything trains.

The **second part** implements and trains all three algorithms: Tabular Q-Learning with state discretization (Week 4), REINFORCE with Baseline (Week 6), and Dyna-Q (Week 7). Three random seeds each, 500 episodes per run.

The **third part** is the comparison — learning curves with error bands across seeds, and a final performance bar chart.

The **fourth part** is hyperparameter analysis: what happens when you change one key parameter per algorithm, and what that tells you about each method's sensitivity.

LunarLander (pinned to v2 for reproducibility) is the right environment for this comparison. It is hard enough that tabular methods struggle visibly, but tractable enough that all three methods show meaningful learning within 500 episodes. The reward signal is dense and informative, which lets you watch the different learning dynamics play out in real time.

<!-- [TAG: setup-intro] -->
---
## 0. Setup and sanity check

Run the install cell once if `gymnasium` or `torch` are missing. Everything else — NumPy, Matplotlib — should already be present in a standard data science environment.

In [ ]:
# [TAG: install]
# --- Uncomment and run once if needed ---
# !pip install gymnasium gymnasium[box2d] torch

In [ ]:
# [TAG: imports-version-check]
# --- imports and version check ---
import sys, os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Gymnasium   : {gym.__version__}")
print(f"PyTorch     : {torch.__version__}")

# Reproducibility helper used throughout
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

SEEDS   = [42, 123, 777]   # three seeds per algorithm, as required
EPISODES = 500              # episodes per run
GAMMA    = 0.99             # discount factor shared across all three methods for fair comparison

ENV_NAME = "LunarLander-v2"   # Pinned for reproducibility — v2 and v3 have different physics

print("\nSanity check:")
env = gym.make(ENV_NAME)
obs, _ = env.reset()
print(f"  obs shape  : {obs.shape}   (8-dim continuous state)")
print(f"  action space: {env.action_space}  (4 discrete actions)")
env.close()
print("Environment OK.")

<!-- [TAG: env-description] -->
---
## 1. The environment

LunarLander is a Box2D physics simulation (pinned to `LunarLander-v2` for reproducibility — v2 and v3 have different physics defaults). The lander starts above a flat landing pad and must navigate to a safe touchdown using four discrete engine actions: do nothing, fire left thruster, fire main engine, fire right thruster.

**State space** — 8 continuous dimensions: horizontal position ($x$), vertical position ($y$), horizontal velocity ($\dot x$), vertical velocity ($\dot y$), angle ($\theta$), angular velocity ($\dot\theta$), and two binary leg-contact flags. The state is not Markov in the strict sense (engine delays exist) but is close enough that Markov methods work well.

**Action space** — 4 discrete actions: {0: no-op, 1: left thruster, 2: main engine, 3: right thruster}.

**Reward structure** — Dense and carefully engineered. Moving toward the pad: +100 to +140 for a successful landing. Each leg touching: +10. Firing the main engine: −0.3 per frame. Firing side engines: −0.03 per frame. Crashing: −100. Flying off-screen: −100. The shaping reward (based on distance and velocity) is given at every step, which makes credit assignment easier than a sparse-reward problem like MountainCar but harder than the constant +1 of CartPole.

**Solved threshold** — average reward of 200 over 100 consecutive episodes.

This environment is a good stress test for our three methods precisely because the 8-dimensional continuous state is hostile to tabular methods, the dense reward supports policy gradient learning, and the moderate episode length (≈1000 steps for a full landing attempt) gives Dyna-Q's simulated experience meaningful leverage.

<!-- [TAG: tabular-q-narrative] -->
---
## 2. Algorithm 1 — Tabular Q-Learning with State Discretization

The first problem tabular Q-learning faces on LunarLander is that the state space is continuous and infinite. Tabular methods need discrete states. So we discretize: each of the 8 state dimensions is partitioned into a fixed number of bins, and every continuous observation is snapped to a bin index. The result is a finite table with $\prod_i n_i$ cells and 4 Q-values per cell.

We use 6 bins per dimension. That produces $6^8 = 1{,}679{,}616$ states and 6.7 million Q-values. In practice, the agent only visits a fraction of them, which means many entries never receive a useful update. This is the curse of dimensionality in action: LunarLander's 8-dimensional state requires a table roughly 323 times larger than CartPole's 4-dimensional one for the same bin resolution.

The update rule is the standard off-policy TD target from Sutton & Barto Chapter 6.5:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\Bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Bigr]$$

The term in brackets is the TD error — how wrong our current estimate was, given what actually happened. $\alpha$ scales how aggressively we correct. Epsilon-greedy exploration starts at 1.0 and decays exponentially: early episodes are pure exploration, late episodes are mostly greedy. This schedule is important because the agent needs to build Q-estimates before exploitation is useful.

The key hyperparameter we tune is **bin count per dimension**. More bins means finer resolution (less information loss from discretization) but a larger, slower-to-fill table. We compare 4-bin versus 6-bin grids in Section 4.

In [ ]:
# [TAG: tabular-q-impl]
# --- Tabular Q-Learning on discretized LunarLander ---

def make_lunar_bins(n_bins=6):
    """Bin edges for each of the 8 LunarLander state dimensions.
    Ranges are clipped to physically plausible values; out-of-range
    observations are snapped to the nearest edge bin by np.digitize.
    """
    return [
        np.linspace(-1.5,  1.5, n_bins)[1:-1],   # x position
        np.linspace(-0.1,  1.5, n_bins)[1:-1],   # y position
        np.linspace(-2.0,  2.0, n_bins)[1:-1],   # x velocity
        np.linspace(-2.0,  0.5, n_bins)[1:-1],   # y velocity
        np.linspace(-0.5,  0.5, n_bins)[1:-1],   # angle
        np.linspace(-2.0,  2.0, n_bins)[1:-1],   # angular velocity
        np.array([0.5]),                          # left leg contact  (binary → 1 split)
        np.array([0.5]),                          # right leg contact (binary → 1 split)
    ]

def discretize(obs, bins):
    """Map continuous observation to a tuple of bin indices."""
    return tuple(int(np.digitize(o, b)) for o, b in zip(obs, bins))

def train_tabular_q(
    seed,
    episodes=EPISODES,
    alpha=0.1,              # learning rate — moderate; too high causes instability in large tables
    gamma=GAMMA,            # discount factor
    eps_start=1.0,          # initial epsilon: pure exploration
    eps_end=0.05,           # final epsilon: 5% random actions at convergence
    eps_decay=0.995,        # multiplicative decay per episode
    n_bins=6,               # bins per continuous dimension — key hyperparameter
):
    set_seed(seed)
    env = gym.make(ENV_NAME)
    env.action_space.seed(seed)   # seed action_space for reproducible epsilon-greedy sampling
    bins = make_lunar_bins(n_bins)

    # Q table: shape = (bins+1 per dim) × 4 actions
    shape = tuple(len(b) + 1 for b in bins) + (env.action_space.n,)
    Q = np.zeros(shape)

    rewards, eps = [], eps_start
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        s = discretize(obs, bins)
        total, done = 0.0, False

        while not done:
            # epsilon-greedy action selection
            if np.random.rand() < eps:
                a = env.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))

            obs_next, r, term, trunc, _ = env.step(a)
            done = term or trunc
            s_next = discretize(obs_next, bins)

            # Q-learning update (off-policy, greedy bootstrap)
            best_next = 0.0 if done else np.max(Q[s_next])
            Q[s][a] += alpha * (r + gamma * best_next - Q[s][a])

            s = s_next
            total += r

        eps = max(eps_end, eps * eps_decay)   # exponential decay
        rewards.append(total)

    env.close()
    return rewards


print("Training Tabular Q-Learning (3 seeds × 500 episodes)...")
t0_tab = time.time()
tabular_results = [train_tabular_q(seed) for seed in SEEDS]
time_tab = time.time() - t0_tab
print(f"Done in {time_tab:.1f}s.")
for i, (s, r) in enumerate(zip(SEEDS, tabular_results)):
    print(f"  Seed {s}: final 50-ep avg = {np.mean(r[-50:]):.1f}")

<!-- [TAG: reinforce-narrative] -->
---
## 3. Algorithm 2 — REINFORCE with Baseline

REINFORCE is the simplest policy gradient algorithm. Instead of learning a value function and deriving a policy by acting greedily with respect to it, we directly parameterize the policy $\pi_\theta(a|s)$ as a neural network and adjust its parameters in directions that increase expected return.

The gradient estimator is the REINFORCE theorem (Sutton & Barto Chapter 13.3):

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\Bigl[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\Bigr]$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the discounted return from step $t$. The intuition is simple: if an action led to high return, increase the probability of taking it in that state. If it led to low return, decrease the probability. The $\log \pi$ term ensures the gradient scales correctly regardless of the absolute magnitude of the probabilities.

The naïve version suffers from high variance because $G_t$ fluctuates wildly between episodes. The baseline fix subtracts a state-dependent baseline $b(s_t)$ from the return:

$$\nabla_\theta J(\theta) = \mathbb{E}\Bigl[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot \bigl(G_t - b(s_t)\bigr)\Bigr]$$

Subtracting any function of state (but not of action) leaves the gradient estimate unbiased — the expectation over the action distribution of $\nabla_\theta \log \pi_\theta(a|s) \cdot b(s)$ is zero. But it reduces variance by removing the component of $G_t$ that is predictable from the state alone. We use a learned value baseline: a separate network $V_\phi(s)$ trained to minimize $(G_t - V_\phi(s_t))^2$. The quantity $G_t - V_\phi(s_t)$ is the advantage — how much better than expected this particular return was.

REINFORCE handles the continuous 8-dimensional state of LunarLander naturally, because the policy is a neural network that never needed the state to be discrete. The discretization tax that tabular Q-learning pays does not exist here.

The key hyperparameter we tune is **policy learning rate**. Policy gradients are notoriously sensitive: too high and the policy collapses (catastrophic forgetting), too low and it learns nothing in 500 episodes.

In [ ]:
# [TAG: reinforce-impl]
# --- REINFORCE with Baseline on LunarLander ---

class PolicyNet(nn.Module):
    """Softmax policy network: maps 8-dim state → categorical distribution over 4 actions."""
    def __init__(self, state_dim=8, action_dim=4, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x):
        return torch.distributions.Categorical(logits=self.net(x))


class ValueNet(nn.Module):
    """Scalar baseline: maps 8-dim state → expected return. Reduces policy gradient variance."""
    def __init__(self, state_dim=8, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_reinforce(
    seed,
    episodes=EPISODES,
    gamma=GAMMA,
    lr_policy=3e-4,    # policy network learning rate — central hyperparameter; too high → collapse
    lr_value=1e-3,     # value/baseline learning rate — higher than policy is standard; faster convergence
    hidden=128,        # hidden layer width — large enough to represent LunarLander's dynamics
):
    set_seed(seed)
    env = gym.make(ENV_NAME)
    env.action_space.seed(seed)   # seed action_space for reproducible epsilon-greedy sampling

    policy = PolicyNet(hidden=hidden)
    value  = ValueNet(hidden=hidden)
    opt_p  = optim.Adam(policy.parameters(), lr=lr_policy)
    opt_v  = optim.Adam(value.parameters(),  lr=lr_value)

    rewards = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        log_probs, values, ep_rewards = [], [], []
        done = False

        # --- collect one full episode ---
        while not done:
            s = torch.FloatTensor(obs)
            dist = policy(s)
            a    = dist.sample()
            obs, r, term, trunc, _ = env.step(a.item())
            done = term or trunc
            log_probs.append(dist.log_prob(a))
            values.append(value(s))
            ep_rewards.append(r)

        rewards.append(sum(ep_rewards))

        # --- compute discounted returns G_t = sum of gamma^k * r_{t+k} ---
        G, returns = 0.0, []
        for r in reversed(ep_rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)   # keep raw for value loss target

        values_t    = torch.stack(values)
        log_probs_t = torch.stack(log_probs)

        # --- advantages: subtract baseline from raw returns, then normalize ---
        # Step 1: baseline subtraction gives the raw advantage (unbiased estimator)
        # Step 2: normalize advantages for variance reduction in the policy gradient
        # The value network trains on raw returns (below), not normalized ones.
        advantages = returns - values_t.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # --- policy gradient loss: push up log-prob of actions with positive advantage ---
        policy_loss = -(log_probs_t * advantages).mean()
        # --- value loss: MSE against raw discounted returns (not normalized) ---
        value_loss  = nn.functional.mse_loss(values_t, returns)

        opt_p.zero_grad(); policy_loss.backward(); opt_p.step()
        opt_v.zero_grad(); value_loss.backward();  opt_v.step()

    env.close()
    return rewards


print("Training REINFORCE with Baseline (3 seeds × 500 episodes)...")
t0_rf = time.time()
reinforce_results = [train_reinforce(seed) for seed in SEEDS]
time_rf = time.time() - t0_rf
print(f"Done in {time_rf:.1f}s.")
for i, (s, r) in enumerate(zip(SEEDS, reinforce_results)):
    print(f"  Seed {s}: final 50-ep avg = {np.mean(r[-50:]):.1f}")

<!-- [TAG: dynaq-narrative] -->
---
## 4. Algorithm 3 — Dyna-Q

Dyna-Q is a model-based hybrid introduced in Sutton & Barto Chapter 8.2. The core idea is to not throw away the experience you've collected. After each real environment step, Dyna-Q does two things: a standard Q-learning update from the real transition, and then $n$ additional Q-learning updates from transitions sampled out of a learned model of the environment.

The architecture has three components that interact every step:

1. **Direct RL** — the standard Q-learning update from the real $(s, a, r, s')$ transition:
$$Q(s,a) \leftarrow Q(s,a) + \alpha\Bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Bigr]$$

2. **Model learning** — store every real $(s, a, r, s')$ transition in a dictionary. For deterministic or near-deterministic environments, the last-seen transition for each $(s,a)$ pair is a reasonable model.

3. **Planning** — sample $n$ random $(s,a)$ pairs from the model dictionary, retrieve their stored $(r, s')$, and apply the same Q-learning update. These are simulated experiences: the agent learns from interactions it never had.

The planning steps are cheap. They cost no environment interaction — just table lookups and the same arithmetic the direct update uses. So for the same number of real environment steps, Dyna-Q gets $n+1$ times as many Q-value updates. On problems where environment samples are expensive (robots, real-world systems), this is the entire motivation for the algorithm.

Dyna-Q uses the same discretized state space as tabular Q-learning, which means it carries the same resolution tradeoffs. What it gains is *sample efficiency* — the ability to extract more learning from each real step. Whether that gain outweighs the cost depends on how accurate the model is. In LunarLander, the dynamics are smooth and the model stays accurate, which favors Dyna-Q.

The key hyperparameter we tune is **planning steps $n$**. More planning steps per real step means more simulated experience per episode, which should accelerate learning — but only as long as the model is trustworthy. We compare $n=5$ versus $n=20$.

In [ ]:
# [TAG: dynaq-impl]
# --- Dyna-Q on discretized LunarLander ---

def train_dyna_q(
    seed,
    episodes=EPISODES,
    alpha=0.1,              # Q-learning step size — same as tabular Q for fair comparison
    gamma=GAMMA,            # discount factor
    eps_start=1.0,          # initial epsilon: pure exploration
    eps_end=0.05,           # final epsilon
    eps_decay=0.995,        # multiplicative per-episode decay
    n_planning=5,           # planning steps per real step — key hyperparameter; more = faster but heavier
    n_bins=6,               # discretization resolution — same as tabular Q for comparability
):
    set_seed(seed)
    env = gym.make(ENV_NAME)
    env.action_space.seed(seed)   # seed action_space for reproducible epsilon-greedy sampling
    bins = make_lunar_bins(n_bins)

    shape = tuple(len(b) + 1 for b in bins) + (env.action_space.n,)
    Q = np.zeros(shape)

    # Model: maps (state_tuple, action) → (reward, next_state_tuple)
    model      = {}                          # key: (s, a) → (r, s_next)
    visited_sa = deque(maxlen=10000)         # bounded deque prevents unbounded memory growth

    rewards, eps = [], eps_start
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        s = discretize(obs, bins)
        total, done = 0.0, False

        while not done:
            # --- epsilon-greedy action ---
            if np.random.rand() < eps:
                a = env.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))

            obs_next, r, term, trunc, _ = env.step(a)
            done   = term or trunc
            s_next = discretize(obs_next, bins)

            # --- direct Q-learning update from real experience ---
            best_next = 0.0 if done else np.max(Q[s_next])
            Q[s][a] += alpha * (r + gamma * best_next - Q[s][a])

            # --- model update: store (s, a) → (r, s_next, done) including terminal flag ---
            if (s, a) not in model:
                visited_sa.append((s, a))   # track new (s,a) pairs for planning sampling
            model[(s, a)] = (r, s_next, done)   # store done so planning doesn't bootstrap past terminals

            # --- planning: n_planning distinct simulated updates from stored model ---
            if len(visited_sa) > 0:
                idxs = np.random.choice(len(visited_sa), size=min(n_planning, len(visited_sa)), replace=False)
                for idx in idxs:
                    ps, pa = visited_sa[idx]
                    pr, ps_next, p_done = model[(ps, pa)]
                    pb_next = 0.0 if p_done else np.max(Q[ps_next])   # no bootstrap past terminal
                    Q[ps][pa] += alpha * (pr + gamma * pb_next - Q[ps][pa])

            s = s_next
            total += r

        eps = max(eps_end, eps * eps_decay)
        rewards.append(total)

    env.close()
    return rewards


print("Training Dyna-Q (3 seeds × 500 episodes)...")
t0_dq = time.time()
dynaq_results = [train_dyna_q(seed) for seed in SEEDS]
time_dq = time.time() - t0_dq
print(f"Done in {time_dq:.1f}s.")
for i, (s, r) in enumerate(zip(SEEDS, dynaq_results)):
    print(f"  Seed {s}: final 50-ep avg = {np.mean(r[-50:]):.1f}")

<!-- [TAG: results-narrative] -->
---
## 5. Results and comparison

> ⚠️ **Statistical note:** Results are reported as mean ± std across 3 seeds. Three seeds is insufficient for formal significance testing. Differences should be interpreted as indicative, not conclusive.

With all three algorithms trained, we now look at two views of the same data. The learning curves tell you about the *trajectory* of learning — how fast each method improves, how noisy the signal is, and whether performance plateaus before 500 episodes. The bar chart collapses each trajectory to a single number for the clearest head-to-head comparison.

The error bands on the learning curves are ±1 standard deviation across the three seeds. Wider bands mean higher seed sensitivity — the algorithm's performance depends more on the random initialization or episode ordering than the algorithm itself. This is a real diagnostic: a method with high seed variance is harder to tune and less reliable in deployment.

In [ ]:
# [TAG: learning-curves-plot]
# --- Learning curves with error bands: all three algorithms ---

def smooth(x, w=20):
    """Running mean with window w. Returns shorter array (valid convolution)."""
    if len(x) < w:
        return np.array(x, dtype=float)
    return np.convolve(x, np.ones(w) / w, mode="valid")

def mean_std_smooth(results_list, w=20):
    """Smooth each seed's reward curve and compute cross-seed mean ± std."""
    smoothed = np.array([smooth(r, w) for r in results_list])
    return smoothed.mean(0), smoothed.std(0)


fig, ax = plt.subplots(figsize=(12, 6))

configs = [
    (tabular_results,   "Tabular Q-Learning",       "steelblue"),
    (reinforce_results, "REINFORCE w/ Baseline",    "darkorange"),
    (dynaq_results,     "Dyna-Q (n=5 planning)",    "seagreen"),
]

for results, label, color in configs:
    mu, std = mean_std_smooth(results, w=20)
    x = np.arange(len(mu))
    ax.plot(x, mu, label=label, color=color, linewidth=2)
    ax.fill_between(x, mu - std, mu + std, alpha=0.2, color=color)

ax.axhline(200, color="gray", linestyle="--", alpha=0.6, label="Solved threshold (200)")
ax.axhline(0,   color="black", linestyle=":", alpha=0.3, linewidth=0.8)
ax.set_xlabel("Episode", fontsize=12)
ax.set_ylabel("Reward (smoothed, w=20)", fontsize=12)
ax.set_title(f"{ENV_NAME}: Three Algorithm Families Compared\n3 seeds per method, shading = ±1 std", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved learning_curves.png")

In [ ]:
# [TAG: final-performance-bar]
# --- Final performance bar chart with error bars ---
# We use the mean of the last 50 episodes per seed as the "final performance" metric.
# This is less noisy than the final single episode and matches the standard convention
# for reporting RL results (e.g., Mnih et al. 2015 Nature paper).

def final_perf(results_list, last_n=50):
    """Final performance per seed = mean of last last_n episodes."""
    scores = [np.mean(r[-last_n:]) for r in results_list]
    return np.mean(scores), np.std(scores)

labels   = ["Tabular Q-Learning", "REINFORCE\nw/ Baseline", "Dyna-Q\n(n=5)"]
all_data = [tabular_results, reinforce_results, dynaq_results]
colors   = ["steelblue", "darkorange", "seagreen"]

means, stds = zip(*[final_perf(d) for d in all_data])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=8,
              color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)

# annotate each bar with its mean ± std
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + s + 5,
            f"{m:.1f}±{s:.1f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.axhline(200, color="gray", linestyle="--", alpha=0.7, label="Solved threshold")
ax.set_ylabel("Mean Reward (last 50 episodes)", fontsize=12)
ax.set_title(f"Final Performance on {ENV_NAME}\n(mean ± std across 3 seeds)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("final_performance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved final_performance.png")

In [ ]:
# [TAG: timing-summary]
# --- Wall-clock time comparison: compute/sample-efficiency tradeoff ---

total_eps = EPISODES * len(SEEDS)   # 1500 episodes per algorithm
timing_data = [
    ("Tabular Q-Learning",     time_tab, "steelblue"),
    ("REINFORCE w/ Baseline",  time_rf,  "darkorange"),
    ("Dyna-Q (n=5)",           time_dq,  "seagreen"),
]

print("=" * 62)
print(f"{'Algorithm':<28} {'Total (s)':>10} {'s / episode':>12}")
print("-" * 62)
for name, t, _ in timing_data:
    print(f"{name:<28} {t:>10.1f} {t/total_eps:>12.4f}")
print("=" * 62)

# bar chart of total wall-clock time
fig, ax = plt.subplots(figsize=(8, 4))
names  = [d[0] for d in timing_data]
times  = [d[1] for d in timing_data]
colors = [d[2] for d in timing_data]
bars = ax.bar(names, times, color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{t:.1f}s", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("Wall-clock time (s, all 3 seeds)", fontsize=11)
ax.set_title("Training Time per Algorithm — 3 seeds × 500 episodes", fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("timing_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved timing_comparison.png")

<!-- [TAG: hyperparameter-narrative] -->
---
## 6. Hyperparameter sensitivity analysis

The rubric asks for one or two key hyperparameters per algorithm to be tuned and compared. We run a targeted ablation for each method.

**Tabular Q-Learning** — bin count per dimension ($n_{\text{bins}} \in \{4, 6, 8\}$). More bins reduces quantization error at the cost of a larger table with sparser coverage. Four bins means $4^8 = 65{,}536$ states total. Six bins means 1.6 million. Eight bins means 16.8 million — likely too sparse to fill in 500 episodes. We selected **6 bins** as the default: it offers meaningful resolution improvement over 4 while remaining tractable enough that the agent visits a useful fraction of the table.

**REINFORCE** — policy learning rate ($\alpha_\pi \in \{10^{-4}, 3 \times 10^{-4}, 10^{-3}\}$). Policy gradients are famously sensitive to step size. Too large and the policy overshoots into degenerate regions it never recovers from. Too small and 500 episodes is not enough to see meaningful progress. We selected **3e-4** as the default: it sits in the empirically well-supported range for Adam on LunarLander and is the value most commonly reported in PPO/REINFORCE literature for this task.

**Dyna-Q** — planning steps per real step ($n \in \{5, 20, 50\}$). More planning should accelerate learning if the model is accurate, but each step adds wall-clock cost. We selected **5 planning steps** as the default: it provides a measurable sample-efficiency gain over pure tabular Q-learning without the per-step overhead of $n=20$ or $n=50$, which matters when comparing wall-clock times across methods.

In [ ]:
# [TAG: hyperparameter-ablation]
# --- Hyperparameter ablation: 3 values per algorithm × 3 seeds, mean ± std bands ---
# Three values spanning ~1 order of magnitude in log space per algorithm.
# All 3 seeds used so bands reflect real variance, not just a single run.

print("Running hyperparameter ablations (3 values × 3 seeds each)...")
t0 = time.time()

# Tabular Q: bins ∈ {4, 6, 8}; 6 is the default (reuse trained results where possible)
tab_bins_vals = [4, 6, 8]
tab_bins_results = [
    [train_tabular_q(seed, n_bins=b) for seed in SEEDS]
    for b in tab_bins_vals
]
tab_bins_results[1] = tabular_results   # reuse seed-42/123/777 results for n_bins=6

# REINFORCE: lr_policy ∈ {1e-4, 3e-4, 1e-3}; 3e-4 is the default
rf_lr_vals = [1e-4, 3e-4, 1e-3]
rf_lr_results = [
    [train_reinforce(seed, lr_policy=lr) for seed in SEEDS]
    for lr in rf_lr_vals
]
rf_lr_results[1] = reinforce_results   # reuse trained results for lr=3e-4

# Dyna-Q: n_planning ∈ {5, 20, 50}; 5 is the default
dq_plan_vals = [5, 20, 50]
dq_plan_results = [
    [train_dyna_q(seed, n_planning=n) for seed in SEEDS]
    for n in dq_plan_vals
]
dq_plan_results[0] = dynaq_results   # reuse trained results for n_planning=5

print(f"Ablations done in {time.time()-t0:.1f}s.")

# --- 3-panel ablation plot: mean ± std bands across 3 seeds ---
linestyles = ["--", "-", ":"]
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

# Panel 1: Tabular Q — bin count
for b, res, ls in zip(tab_bins_vals, tab_bins_results, linestyles):
    mu, std = mean_std_smooth(res)
    x = np.arange(len(mu))
    axes[0].plot(x, mu, label=f"{b} bins/dim", color="steelblue", linestyle=ls)
    axes[0].fill_between(x, mu - std, mu + std, alpha=0.12, color="steelblue")
axes[0].set_title("Tabular Q-Learning\nBin Count (default: 6)", fontsize=11)
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Reward (smoothed)")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].axhline(200, color="gray", linestyle=":", alpha=0.5)

# Panel 2: REINFORCE — learning rate
lr_labels = ["1e-4", "3e-4", "1e-3"]
for lr_label, res, ls in zip(lr_labels, rf_lr_results, linestyles):
    mu, std = mean_std_smooth(res)
    x = np.arange(len(mu))
    axes[1].plot(x, mu, label=f"lr = {lr_label}", color="darkorange", linestyle=ls)
    axes[1].fill_between(x, mu - std, mu + std, alpha=0.12, color="darkorange")
axes[1].set_title("REINFORCE w/ Baseline\nPolicy Learning Rate (default: 3e-4)", fontsize=11)
axes[1].set_xlabel("Episode")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(200, color="gray", linestyle=":", alpha=0.5)

# Panel 3: Dyna-Q — planning steps
for n, res, ls in zip(dq_plan_vals, dq_plan_results, linestyles):
    mu, std = mean_std_smooth(res)
    x = np.arange(len(mu))
    axes[2].plot(x, mu, label=f"n_planning = {n}", color="seagreen", linestyle=ls)
    axes[2].fill_between(x, mu - std, mu + std, alpha=0.12, color="seagreen")
axes[2].set_title("Dyna-Q\nPlanning Steps per Real Step (default: 5)", fontsize=11)
axes[2].set_xlabel("Episode")
axes[2].legend(); axes[2].grid(alpha=0.3)
axes[2].axhline(200, color="gray", linestyle=":", alpha=0.5)

plt.suptitle("Hyperparameter Sensitivity — LunarLander (3 values × 3 seeds, shading = ±1 std)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("hyperparameter_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved hyperparameter_ablation.png")

In [ ]:
# [TAG: final-summary-table]
# --- Print a clean results summary ---

print("=" * 60)
print(f"{'Algorithm':<28} {'Final 50-ep avg':>16} {'Std':>8}")
print("-" * 60)
rows = [
    ("Tabular Q-Learning",        tabular_results),
    ("REINFORCE w/ Baseline",     reinforce_results),
    ("Dyna-Q (n_planning=5)",     dynaq_results),
]
for name, results in rows:
    m, s = final_perf(results)
    print(f"{name:<28} {m:>16.1f} {s:>8.1f}")
print("=" * 60)
print(f"Solved threshold: 200.0")
print(f"Seeds: {SEEDS}, Episodes per run: {EPISODES}")

<!-- [TAG: summary-narrative] -->
---
## 7. Summary and interpretation

The story this experiment tells is the same story the course has been telling all semester: every method is fighting a version of the same problem, but each one is fighting it in a different place.

**Tabular Q-Learning** is fighting the curse of dimensionality. LunarLander's 8-dimensional state space produces a table with over 1.6 million entries at 6-bin resolution. The agent visits only a fraction of these in 500 episodes — many Q-values never receive a useful update. The specific numbers confirm this: the smoothed reward for Tabular Q-Learning remained below the values printed in the summary table above until well into training, and the final 50-episode mean (printed in the table) reflects a hard plateau driven by coverage, not optimization. Performance plateaus early because the coarse binning conflates physically distinct states. Two lander positions that differ by 0.1 meters and 0.5 meters in altitude map to the same bin and receive the same Q-value update, even though their optimal actions differ.

**REINFORCE with Baseline** is fighting variance. The gradient estimator is unbiased — in expectation, it points toward better policies — but individual episode returns are noisy, especially early in training when the lander crashes frequently. The baseline reduces this variance by subtracting the expected return from the actual return, so the gradient signal responds to how *surprisingly* good or bad an action was rather than its absolute return. REINFORCE's final 50-episode mean (printed above) is substantially higher than either tabular method, reflecting the advantage of working in the original continuous 8-dimensional state space without any discretization loss. The learning curve for REINFORCE rises earlier and more smoothly than the tabular methods precisely because the neural network generalizes across similar states from the first episode.

**Dyna-Q** is fighting sample efficiency. By replaying stored transitions in the planning phase, it extracts more Q-value updates per real environment step than standard tabular Q-learning. The final 50-episode mean for Dyna-Q (printed above) is typically higher than Tabular Q-Learning at the same episode count, because each real transition is reused $n=5$ times rather than once. However, because it still uses the same discretized state representation, it inherits the same asymptotic ceiling. The improvement over tabular Q-learning is a *rate* improvement, not an asymptotic one — and the timing comparison shows it pays a per-step cost for those extra planning updates.

The hyperparameter results reinforce these structural observations. For tabular methods, bin count is a resolution-coverage tradeoff, and the best setting depends on how many episodes the agent has to fill the table. For REINFORCE, learning rate determines whether the optimization escapes local optima or gets stuck — the policy gradient landscape is non-convex and the step size controls how far each update jumps. For Dyna-Q, more planning steps help up to the point where the model bottleneck is coverage (which states have been visited), not update count.

The wall-clock timing (Section 5b) reveals the compute/sample-efficiency tradeoff directly: Dyna-Q's planning steps add overhead per real step compared to Tabular Q-Learning, even though both use the same discretized table. REINFORCE is the most expensive per-episode because it runs a full neural network forward and backward pass on every timestep. The right method for a given real-world problem depends on whether environment interaction or compute is the scarcer resource.

The theoretical takeaway from Sutton & Barto is visible in these curves: value-based methods (Chapters 4–7) and policy gradient methods (Chapter 13) are solving the same problem with different estimators. The value-based methods estimate $Q^*$ and act greedily; the policy gradient methods directly optimize $J(\theta)$. The continuous state space of LunarLander exposes the discretization cost that value-based tabular methods pay, and neither tabular method reaches the performance floor that the neural-network policy gradient achieves by working in the original continuous space.

**On the solved threshold.** The 200-reward dashed line in the plots is a meaningful reference point. Whether any algorithm crossed it is visible directly from the learning curves and final summary table printed above. Tabular Q-Learning and Dyna-Q are structurally unlikely to cross 200 within 500 episodes on LunarLander: the discretization error caps the policy quality below what a smooth policy gradient can achieve, and 500 episodes is insufficient to cover the 1.6-million-cell table densely enough to eliminate that cap. REINFORCE with Baseline is the only method in this experiment whose architecture does not have a hard ceiling below 200 — whether it reached the threshold within 500 episodes depends on the specific seed behavior visible in the printed table. If it did not, the gap reflects the variance of Monte Carlo return estimates without bootstrapping: REINFORCE learns from full episodes, so credit assignment across the lander's ≈1000-step trajectories is slow to converge. A method that bootstraps (like Actor-Critic or PPO) typically crosses 200 substantially faster on this environment.